In [227]:
print("Hello world")
print("Hello world")
print("Hello world")

Hello world
Hello world
Hello world


In [228]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from scipy.stats import norm

In [229]:
#Spot
S0 = 1
#Strike
K = 0.1
#Volatilité
sig = 1
#taux-sans-risque
r = 0.017
#Maturité
T = 1
#dividende
q = 1

In [230]:
#Tirage aléatoir

def lcg(seed, a, c, m, n):

    x = seed
    xi = []

    for i in range (n):
        x = (a*x + c)%m
        xi.append(x/m)

    return xi

In [231]:
#Box-Muller

def BoxMuller(U):
    Z = []
    for u in range(0, len(U)-1, 2):
        R = -2*math.log(U[u])
        V = 2*math.pi*U[u+1]
        Z1, Z2 = math.sqrt(R)*math.cos(V), math.sqrt(R)*math.sin(V)
        Z.append(Z1)
        Z.append(Z2)
    return Z

In [232]:
#Création d'une matrice de valeur suivant une loi normal

def MatriceBM(N,M):
    #print(f"Pour {N*M} tirrage")
    U = lcg(12345, 16807, 1153, (2**31) - 1, N*M+1)
    #print(U)
    Z = BoxMuller(U)
    #print(Z)

    Z = np.array(Z[:N*M])
    Z = Z.reshape(N, M)
    #print(Z)

    return Z

MatriceBM(2,5)


array([[ 1.19284212, -1.80308832,  0.85981498, -0.46631121,  0.78427378],
       [ 0.20353608, -0.29137584,  1.7290058 , -1.35478422,  0.58407968]])

In [233]:
#Call et PUT

def CALL(K,St):
    return max(St-K, 0)

def PUT(K,St):
    return max(K-St, 0)

In [234]:
#Monte Carlo par Black-Scholes

def MCeuroBS(K, r, q, T, sig, S0):
    d1 = (math.log(S0/K)+(r-q+0.5*(sig*sig))*T)/(sig*math.sqrt(T))
    d2 = d1 - sig*math.sqrt(T)

    C = S0*math.exp(-q*T)*norm.cdf(d1) - K*math.exp(-r*T)*norm.cdf(d2)
    P = K*math.exp(-r*T)*norm.cdf(-d2) - S0*math.exp(-q*T)*norm.cdf(-d1)

    return C, P

In [235]:
#Simulation des chemins (prix du produit)
def SimChemin(N, M, T):

    Z = MatriceBM(N,M)

    dt= T/M
    nudt = (r - q - 0.5 * sig**2) * dt
    sigsdt = sig * math.sqrt(dt)

    S = np.empty((N, M + 1), dtype=float)
    S[:, 0] = S0

    for i in range(N):
        for j in range(M):
            S[i, j+1] = S[i, j] * math.exp(nudt + sigsdt * Z[i, j])

    #print(S)
    return S


In [236]:
#Valuing American Options by Simulation: A Simple Least-Squares Approach

def ValAmerican(N, M, deg, T, tdm, type):
    S = SimChemin(N, M, T)

    #print("Simulation des chemins:\n",S)

    dt= T/M
    CF = []

    actu = math.exp(-r * dt)

    for i in range(N):
        if type:
            CF.append(CALL(K,S[i, M]))
        else:
            CF.append(PUT(K,S[i, M]))

    for j in range(M-1):

        X, Y = [], []

        for i in range(N):
            CF[i] *= actu
            if type:
                payoff_now = CALL(K, S[i, M-j-1])
            else:
                payoff_now = PUT(K, S[i, M-j-1])
            if tdm:
                if payoff_now > 0:        # garder seulement ITM
                    X.append(S[i, M-j-1])
                    Y.append(CF[i])
            else:
                X.append(S[i, M-j-1])      # toutes les trajectoires
                Y.append(CF[i])
        #print(CF)
        #print(X)


        # Construire la matrice des polynômes : [1, X, X², ...]
        A = np.vander(X, deg + 1, increasing=True)
        # Résoudre la régression par moindres carrés
        beta, *_ = np.linalg.lstsq(A, Y, rcond=None)

        C_hat_all = np.vander(S[:, M-j-1], deg + 1, increasing=True) @ beta
        #print(C_hat_all)

        for i in range(N):
            if type:
                if CALL(K, S[i, M-j-1]) > C_hat_all[i]:
                    CF[i] = CALL(K, S[i, M-j-1])
            else:
                if PUT(K, S[i, M-j-1]) > C_hat_all[i]:
                    CF[i] = PUT(K, S[i, M-j-1])

        #print(CF)

    somme = 0
    for i in range(N):
        CF[i] = CF[i] * actu
        somme += CF[i]
    #print(CF)


    V0 = (1/N)*somme
    print(f"Type  call->True ou un put->False:  {type}\nTrajectoires dans la monnaie :  {tdm}\nPricin du call américain :{V0}")
    return V0, CF

In [237]:
#Intervale de confiance

def IC(V0, lamb, N, CF):

    lamb1 = 1-lamb/2

    s2 = 0
    for i in range(N):
        s2 += (CF[i] - V0)**2
    s2 *= 1/(N-1)

    IC_low = V0 - (1-lamb1)* math.sqrt(s2/N)
    IC_high = V0 + (1-lamb1)* math.sqrt(s2/N)

    print(f"IC({lamb*100}%) = [{IC_low} , {IC_high}]")

    return IC_high, IC_low

In [238]:
#degré du polynôme de régression
deg = 2
#Nombre de simulation
N = 100
#Combien de fois on peut décider d'exercer l'option
M = 30
#Si on veut prendre l’ensemble des trajectoires ou uniquement les trajectoires dans la monnaie
tdm = True
#Si on veut un "call"->True ou un "put"->False
type = True
#Intervale de confiance entre 0 et 1
lamb = 0.95

V0, CF = ValAmerican(N, M, deg, T, tdm, type)
IC_high, IC_low = IC(V0, lamb, N, CF)
tdm = False
V0, CF = ValAmerican(N, M, deg, T, tdm, type)
IC_high, IC_low = IC(V0, lamb, N, CF)


tdm = True
type = False

V0, CF = ValAmerican(N, M, deg, T, tdm, type)
IC_high, IC_low = IC(V0, lamb, N, CF)
tdm = False
V0, CF = ValAmerican(N, M, deg, T, tdm, type)
IC_high, IC_low = IC(V0, lamb, N, CF)

Type  call->True ou un put->False:  True
Trajectoires dans la monnaie :  True
Pricin du call américain :0.8689166589683301
IC(95.0%) = [0.8601187110415995 , 0.8777146068950608]
Type  call->True ou un put->False:  True
Trajectoires dans la monnaie :  False
Pricin du call américain :0.8689166589683301
IC(95.0%) = [0.8601187110415995 , 0.8777146068950608]
Type  call->True ou un put->False:  False
Trajectoires dans la monnaie :  True
Pricin du call américain :0.002281662334108323
IC(95.0%) = [0.0017347816882143925 , 0.0028285429800022533]
Type  call->True ou un put->False:  False
Trajectoires dans la monnaie :  False
Pricin du call américain :0.005142744030503883
IC(95.0%) = [0.004619939622429254 , 0.0056655484385785115]


In [239]:
Ce, Pe = MCeuroBS(K, r, q, T, sig, S0)
print("Pricing call européen par Black-Scholes : ",Ce)
print("Pricing put européen par Black-Scholes : ",Pe)

Pricing call européen par Black-Scholes :  0.27718094609593746
Pricing put européen par Black-Scholes :  0.0076158733879860985
